In [ ]:
import openmeteo_requests
import requests_cache
import pandas as pd
import time
from retry_requests import retry
import os

cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

cities = {
    "Karachi":    (24.8607, 67.0011),
    "Lahore":     (31.5204, 74.3587),
    "Islamabad":  (33.6844, 73.0479),
    "Rawalpindi": (33.5651, 73.0169),
    "Faisalabad": (31.4180, 73.0790),
    "Multan":     (30.1575, 71.5249),
    "Peshawar":   (34.0151, 71.5249),
    "Quetta":     (30.1798, 66.9750)
}

SAVE_DIR = "city_cache"
os.makedirs(SAVE_DIR, exist_ok=True)

def fetch_with_retry(city, lat, lon, max_retries=5):
    # Skip if already saved
    save_path = os.path.join(SAVE_DIR, f"{city}.csv")
    if os.path.exists(save_path):
        print(f"  Already saved, loading from disk.")
        return pd.read_csv(save_path)

    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": "1960-01-01",
        "end_date": "2024-12-31",
        "daily": [
            "relative_humidity_2m_max",
            "relative_humidity_2m_min",
            "precipitation_sum",
            "et0_fao_evapotranspiration"
        ],
        "timezone": "Asia/Karachi"
    }

    for attempt in range(max_retries):
        try:
            responses = openmeteo.weather_api(url, params=params)
            r = responses[0]
            daily = r.Daily()
            df = pd.DataFrame({
                "time":   pd.date_range(
                              start=pd.to_datetime(daily.Time(), unit="s"),
                              periods=daily.Variables(0).ValuesAsNumpy().shape[0],
                              freq="D"
                          ),
                "rh_max": daily.Variables(0).ValuesAsNumpy(),
                "rh_min": daily.Variables(1).ValuesAsNumpy(),
                "prcp":   daily.Variables(2).ValuesAsNumpy(),
                "et0":    daily.Variables(3).ValuesAsNumpy(),
                "city":   city
            })
            df.to_csv(save_path, index=False)  # save immediately
            return df
        except Exception as e:
            if "limit exceeded" in str(e).lower():
                wait = 60 * (attempt + 1)
                print(f"  Rate limited. Waiting {wait}s before retry {attempt+1}/{max_retries}...")
                time.sleep(wait)
            else:
                raise

    raise RuntimeError(f"Failed to fetch {city} after {max_retries} retries")

all_dfs = []
for city, (lat, lon) in cities.items():
    print(f"Fetching {city}...")
    df = fetch_with_retry(city, lat, lon)
    all_dfs.append(df)
    print(f"  Got {len(df)} rows")
    time.sleep(20)

humidity_df = pd.concat(all_dfs, ignore_index=True)
humidity_df.to_csv("pakistan_humidity_daily.csv", index=False)
print(f"\nDone. Saved pakistan_humidity_daily.csv — {len(humidity_df)} total rows")